# Week 2 Concepts — LLM Internals, Audio AI, Sandboxing

Runnable companion notebook for Days 1-4, organized by day to match the daily notes.
Every code cell is heavily commented to explain *why* each step happens, and every
line that produces or transforms a tensor/array carries an explicit shape comment.

**What actually ran in this sandbox vs. what didn't:** the `tiktoken`-based tokenization
examples, every from-scratch `numpy`/`scipy` implementation (BPE, self-attention, the
KV-cache memory simulation, the log-Mel spectrogram pipeline, retrieval metrics, the
`subprocess`/`resource` sandboxing demo including the RLIMIT_AS platform-failure case),
the TF-IDF bi-encoder-style demo, and the `pyttsx3` OS-level TTS call were all
independently executed here and their real output is reproduced in the markdown cells
around them. Cells that call `sentence-transformers` (`SentenceTransformer`, `CrossEncoder`),
`whisper`, or a hosted `Sandbox` SDK are **not executed in this environment** — no GPU, no
torch, multi-hundred-MB model downloads, and no real sandbox provider credentials are
practical here — but are written against the real, current API and are clearly marked below.

## Day 1: Tokenization, Attention, and the KV Cache

Four pieces, each verified by running it: a from-scratch BPE trainer, real `tiktoken`
token counts for English vs. Korean, a from-scratch numpy self-attention pass (with
causal masking), and a KV-cache memory-growth simulation using real 7B-model-scale
numbers.

In [ ]:
# --- From-scratch BPE trainer over a tiny toy corpus ---
# Same example as Sennrich et al.'s original BPE paper: 4 words, each split into
# characters plus an end-of-word marker '_' (so 'est' at a word boundary is a
# different symbol than 'est' mid-word until they're proven to behave the same).
import collections

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}  # word -> frequency

def word_to_symbols(word):
    return list(word) + ["_"]

# vocab: tuple-of-symbols -> frequency, e.g. ('l', 'o', 'w', '_') -> 5
vocab = {tuple(word_to_symbols(w)): f for w, f in corpus.items()}

def get_pair_counts(vocab):
    # count every adjacent symbol pair across the vocab, weighted by word frequency
    pairs = collections.Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    # replace every occurrence of `pair` with a single merged symbol
    a, b = pair
    merged = a + b
    new_vocab = {}
    for symbols, freq in vocab.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = new_vocab.get(tuple(new_symbols), 0) + freq
    return new_vocab

print("initial vocab (word -> symbol sequence):")
for symbols, freq in vocab.items():
    print(f"  {symbols}  freq={freq}")

num_merges = 6
for step in range(num_merges):
    pairs = get_pair_counts(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)  # most frequent adjacent pair -> next merge rule
    vocab = merge_vocab(best, vocab)
    print(f"\nstep {step + 1}: merge {best} (seen {pairs[best]}x) -> '{best[0] + best[1]}'")
    for symbols, freq in vocab.items():
        print(f"  {symbols}  freq={freq}")
# Expected (verified) merge order: ('e','s') -> ('es','t') -> ('est','_') -> ('l','o') -> ('lo','w') -> ('n','e')

In [ ]:
# --- Real tiktoken token counts: English vs. Korean, same rough meaning ---
# Requires `pip install tiktoken` (pure-Python-ish, installs in seconds, no GPU needed).
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # the GPT-4-family byte-level BPE tokenizer
en = "The quarterly report is ready for review."
ko = "분기 보고서 검토 준비가 완료되었습니다."

en_ids = enc.encode(en)  # -> list[int]
ko_ids = enc.encode(ko)  # -> list[int]
print(len(en_ids), "tokens for English:", en_ids)
print(len(ko_ids), "tokens for Korean:", ko_ids)
print(f"chars/token  EN={len(en)/len(en_ids):.2f}  KO={len(ko)/len(ko_ids):.2f}")
# Verified real output: 8 tokens (EN) vs 20 tokens (KO) -- 2.5x more tokens for
# roughly the same content, because the merge vocabulary was learned mostly on
# English-dominated training text.

# Decode each Korean token back to raw bytes to see *why*: some syllables never
# earned their own merged token and fall all the way back to individual UTF-8 bytes.
print("\nKorean token -> raw bytes:")
for tid in ko_ids:
    b = enc.decode_single_token_bytes(tid)
    print(f"  id={tid:>6}  bytes={b!r}  ({len(b)} byte{'s' if len(b) != 1 else ''})")
# Verified: the syllable '토' (in '검토') is NOT one token -- it splits into three
# separate single-byte tokens (b'\xed', b'\x86', b'\xa0'), the three raw UTF-8
# bytes that make up that one character.

In [ ]:
# --- From-scratch scaled dot-product self-attention, with shapes at every step ---
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # numerical stability, doesn't change the result
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

np.random.seed(0)
tokens = ["The", "cat", "sat", "on", "mat"]
seq_len, d_model = len(tokens), 8  # a real 7B model uses d_model=4096; 8 keeps printed numbers readable

X = np.random.randn(seq_len, d_model)  # shape: (5, 8) -- token embeddings + position info, already summed

# Learned in a real model; random-but-fixed here so the mechanism is visible without training.
W_q = np.random.randn(d_model, d_model) * 0.1  # shape: (8, 8)
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = X @ W_q  # shape: (5, 8) -- "what each token is looking for"
K = X @ W_k  # shape: (5, 8) -- "what each token advertises about itself"
V = X @ W_v  # shape: (5, 8) -- "what each token contributes if attended to"

scores = (Q @ K.T) / np.sqrt(d_model)  # shape: (5, 5) -- scores[i, j] = query_i . key_j, scaled
weights = softmax(scores, axis=-1)     # shape: (5, 5), each row sums to 1.0 -- attention weights
context = weights @ V                  # shape: (5, 8) -- new, context-mixed representation per token

print("attention weights (rows=query token, cols=key token):")
print(np.round(weights, 2))
print("row sums (should all be 1.0):", weights.sum(axis=1))
print("context shape:", context.shape)

In [ ]:
# --- Causal masking: token i may only attend to tokens <= i (needed for generation) ---
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)  # shape: (5, 5), True strictly above diagonal
masked_scores = np.where(causal_mask, -np.inf, scores)  # future positions get -inf score
causal_weights = softmax(masked_scores, axis=-1)        # shape: (5, 5) -- upper triangle collapses to exactly 0

print("causal attention weights (upper triangle must be exactly 0):")
print(np.round(causal_weights, 2))

In [ ]:
# --- KV cache memory footprint, real 7B-model-class numbers ---
# Config inspired by a real ~7B-parameter transformer: 32 layers, 32 attention heads,
# head_dim=128 (so d_model = 32*128 = 4096), weights stored in fp16 (2 bytes/value).
n_layers, n_heads, head_dim, dtype_bytes = 32, 32, 128, 2

def kv_cache_bytes(seq_len, n_kv_heads, batch_size=1):
    # cache shape per layer: (batch, n_kv_heads, seq_len, head_dim), one tensor each for K and V
    per_token_per_layer = 2 * n_kv_heads * head_dim * dtype_bytes  # x2 for storing both K and V
    return batch_size * n_layers * seq_len * per_token_per_layer

print(f"{'seq_len':>8}  {'MHA (32 kv heads)':>20}  {'GQA (8 kv heads)':>18}  {'MQA (1 kv head)':>16}  reduction")
for seq_len in [128, 1024, 8192, 32768]:
    mha = kv_cache_bytes(seq_len, n_kv_heads=32)  # standard multi-head attention
    gqa = kv_cache_bytes(seq_len, n_kv_heads=8)   # grouped-query attention: 8 kv heads shared by 32 q heads
    mqa = kv_cache_bytes(seq_len, n_kv_heads=1)   # multi-query attention: a single shared kv head
    print(f"{seq_len:>8}  {mha/1e6:>17.1f} MB  {gqa/1e6:>15.1f} MB  {mqa/1e6:>13.1f} MB  {mha/gqa:.1f}x smaller w/ GQA")

print("\nKV cache growth across a single generation (MHA, batch=1):")
for n_generated in [1, 10, 100, 1000]:
    b = kv_cache_bytes(n_generated, n_kv_heads=32)
    print(f"  after {n_generated:>4} tokens: {b/1e6:8.2f} MB")
# Verified: at seq_len=32768 the MHA cache alone is ~17.2GB *per request* -- comparable
# to the size of the model's own weights -- which is exactly why GQA/MQA (here, a clean
# 4x reduction from 32 -> 8 kv heads) matter for serving long-context requests at scale.

## Day 2: Embedding Models and Reranking

A TF-IDF-based demonstration of the bi-encoder failure mode (purely lexical, so it isolates
the structural weakness cleanly), the real `sentence-transformers` API for an actual neural
bi-encoder and cross-encoder (not executed here), and a from-scratch Recall@k / MRR
evaluation over a toy ranked-retrieval result set.

In [ ]:
# --- TF-IDF as a runnable stand-in for "encode independently, then compare vectors" ---
# TF-IDF does the same *shape* of computation as a bi-encoder (encode each text on its
# own into a fixed vector, compare with cosine similarity) but with purely lexical
# features and zero learned synonymy -- which makes the failure mode maximally visible.
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

query = "how do I reset my account password"

docs = [
    # actually relevant, but paraphrased -- shares almost no literal words with the query
    "Forgot your login credentials? Use the 'Trouble signing in' link on the sign-in page to create a new one.",
    # wrong topic (a smart-lock manual), but reuses 'account', 'password', 'reset' verbatim
    "The account password reset button on our smart lock clears a stored 4-digit code back to its factory default.",
]

vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform([query] + docs).toarray()  # shape: (3, vocab_size)
q_vec, d_vecs = tfidf[0], tfidf[1:]                          # q_vec: (vocab_size,); d_vecs: (2, vocab_size)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

for i, doc in enumerate(docs):
    print(f"sim={cosine_sim(q_vec, d_vecs[i]):.4f}  {doc[:65]}...")
# Verified real output: the genuinely relevant paraphrase scores 0.0000 (zero literal
# overlap), while the wrong-topic smart-lock doc scores 0.2027 purely from reusing
# 'account'/'password'/'reset' verbatim -- exactly the failure mode a bi-encoder trained
# without enough hard negatives will still partially exhibit, just less severely.

In [ ]:
# --- Real sentence-transformers API (current signatures) -- NOT executed in this sandbox ---
# No torch/sentence-transformers installed here, and no GPU; shown for the correct,
# current method names and shapes, matching the real neural equivalent of the cell above.
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
query = "how do I reset my account password"
docs = [
    "Forgot your login credentials? Use the 'Trouble signing in' link on the sign-in page to create a new one.",
    "The account password reset button on our smart lock clears a stored 4-digit code back to its factory default.",
]

q_vec = bi_encoder.encode(query)   # shape: (384,) for this model
d_vecs = bi_encoder.encode(docs)   # shape: (2, 384)
bi_scores = d_vecs @ q_vec / (np.linalg.norm(d_vecs, axis=1) * np.linalg.norm(q_vec))  # shape: (2,)

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
cross_scores = cross_encoder.predict([(query, d) for d in docs])  # shape: (2,) -- one scalar per pair

In [ ]:
# --- Recall@k and MRR, from scratch, over a toy ranked-retrieval result set ---
import numpy as np

relevant_doc = {"q1": "d7", "q2": "d3", "q3": "d9", "q4": "d1", "q5": "d5"}
ranked_results = {
    "q1": ["d2", "d7", "d4", "d9", "d1"],   # relevant d7 found at rank 2
    "q2": ["d3", "d8", "d1", "d2", "d4"],   # relevant d3 found at rank 1
    "q3": ["d2", "d4", "d1", "d8", "d6"],   # relevant d9 missing entirely -> miss
    "q4": ["d5", "d2", "d1", "d3", "d9"],   # relevant d1 found at rank 3
    "q5": ["d5", "d1", "d2", "d3", "d4"],   # relevant d5 found at rank 1
}

def recall_at_k(ranked, relevant, k):
    hits = sum(relevant[q] in docs[:k] for q, docs in ranked.items())
    return hits / len(ranked)

def mrr(ranked, relevant):
    reciprocal_ranks = []
    for q, docs in ranked.items():
        if relevant[q] in docs:
            reciprocal_ranks.append(1.0 / (docs.index(relevant[q]) + 1))  # 1-indexed rank
        else:
            reciprocal_ranks.append(0.0)  # relevant doc never retrieved
    return np.mean(reciprocal_ranks)

for k in [1, 3, 5]:
    print(f"Recall@{k}: {recall_at_k(ranked_results, relevant_doc, k):.2f}")
print(f"MRR: {mrr(ranked_results, relevant_doc):.3f}")
# Verified: Recall@1=0.40, Recall@3=Recall@5=0.80 (q3's relevant doc never appears in
# the candidate list at all -- a stage-1 retrieval miss no reranker can fix), MRR=0.567.

## Day 3: Audio AI — ASR and TTS

A from-scratch waveform -> STFT -> log-Mel spectrogram pipeline (the actual preprocessing
step every speech transformer, Whisper included, runs internally), built and run with only
`numpy`/`scipy` -- no audio library needed. Followed by the real, current Whisper API
(not executed here) and an OS-level TTS fallback.

In [ ]:
# --- From-scratch waveform -> STFT -> log-Mel spectrogram ---
import numpy as np
from scipy.signal import stft

np.random.seed(42)  # fixes the noise term below so the printed output is reproducible
sample_rate = 16000  # Hz -- the standard ASR input rate, including Whisper's
duration_s = 1.0
t = np.linspace(0, duration_s, int(sample_rate * duration_s), endpoint=False)  # shape: (16000,)

# Synthetic "voice-like" signal: a fundamental + two harmonics (like a sung vowel),
# amplitude-modulated to fake a speech-like on/off envelope, plus a little noise.
f0 = 150  # Hz, a plausible pitch
signal = (1.0 * np.sin(2 * np.pi * f0 * t)
          + 0.5 * np.sin(2 * np.pi * 2 * f0 * t)
          + 0.3 * np.sin(2 * np.pi * 3 * f0 * t))
envelope = 0.5 * (1 + np.sin(2 * np.pi * 2 * t - np.pi / 2))
signal = signal * envelope + 0.02 * np.random.randn(len(t))  # shape: (16000,) float64, raw amplitude samples
print("waveform shape:", signal.shape, signal.dtype)

# --- STFT: slide a 25ms window across the waveform every 10ms ---
win_length = int(0.025 * sample_rate)  # 400 samples = 25ms
hop_length = int(0.010 * sample_rate)  # 160 samples = 10ms
freqs, frame_times, Zxx = stft(signal, fs=sample_rate, nperseg=win_length, noverlap=win_length - hop_length)
magnitude = np.abs(Zxx)  # shape: (201, 101) -> (freq_bins, time_frames); 201 = win_length//2 + 1
print("STFT magnitude shape:", magnitude.shape, "-> (freq_bins, time_frames)")

In [ ]:
# --- Mel filterbank, built from the standard formulas (no librosa in this sandbox) ---
def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

n_mels = 40
n_fft_bins = magnitude.shape[0]  # 201
mel_points = np.linspace(hz_to_mel(0), hz_to_mel(sample_rate / 2), n_mels + 2)  # shape: (42,) mel-scale boundaries
bin_points = np.floor((n_fft_bins - 1) * 2 * mel_to_hz(mel_points) / sample_rate).astype(int)  # -> FFT bin indices

filterbank = np.zeros((n_mels, n_fft_bins))  # shape: (40, 201) -- triangular filters, one row per mel band
for m in range(1, n_mels + 1):
    left, center, right = bin_points[m - 1], bin_points[m], bin_points[m + 1]
    for k in range(left, center):
        if center != left:
            filterbank[m - 1, k] = (k - left) / (center - left)      # rising edge of the triangle
    for k in range(center, right):
        if right != center:
            filterbank[m - 1, k] = (right - k) / (right - center)    # falling edge of the triangle

with np.errstate(all="ignore"):  # benign BLAS warning on some platforms for zero-heavy matmuls
    mel_spec = filterbank @ magnitude  # shape: (40, 201) @ (201, 101) -> (40, 101)
log_mel_spec = np.log(mel_spec + 1e-6)  # shape: (40, 101) -- log-compressed, matches perceived loudness

print("log-mel spectrogram shape:", log_mel_spec.shape, "-> (n_mels, time_frames)")
print("value range:", round(float(log_mel_spec.min()), 2), "to", round(float(log_mel_spec.max()), 2))
# Verified: 1 second of raw audio (16000 numbers) becomes a (40, 101) array -- this is
# the actual object a speech transformer's encoder consumes, never the raw samples.

In [ ]:
# --- Real Whisper API (current signatures) -- NOT executed in this sandbox ---
# whisper isn't installed here, and its model weights (100s of MB) + inference are
# impractical in this environment; shown for the correct, current openai-whisper API.
import whisper

model = whisper.load_model("base")

# Without a domain hint, a rare product name/jargon term can get transcribed as the
# nearest phonetically similar common word (a real, systematic failure mode).
result_plain = model.transcribe("standup_notes.wav")
print("plain:", result_plain["text"])

# initial_prompt conditions the decoder on expected vocabulary before it starts
# generating, biasing (not guaranteeing) decoding toward the right token when ambiguous.
result_hinted = model.transcribe(
    "standup_notes.wav",
    initial_prompt="Kubernetes, ingress, staging, deployment pipeline",
)
print("hinted:", result_hinted["text"])

In [ ]:
# --- OS-level TTS fallback: zero extra dependency, useful as a no-setup baseline ---
import pyttsx3

engine = pyttsx3.init()
engine.say("The nightly backup completed with no errors.")
engine.runAndWait()

## Day 4: Code-Execution Sandboxing for Agents

The generic create/run/files/kill lifecycle a hosted sandbox SDK exposes (schematic --
provider method names vary), followed by a verified local `subprocess` + `resource`
pattern -- including what happens when it meets a CPU-bound infinite loop, and where
it silently fails on this platform -- explicitly marked as a speed bump, never a
security boundary.

In [ ]:
# Schematic example of a hosted sandbox SDK's typical lifecycle.
# Illustrative only -- exact method names differ across providers, but create -> run ->
# move files in/out -> tear down, with an explicit network toggle, is close to universal.

sandbox = Sandbox.create(timeout=60, network_access=False)

output = sandbox.run_code("import statistics; print(statistics.mean([3, 7, 9, 12]))")

sandbox.files.write("/tmp/report.csv", csv_bytes)
sandbox.commands.run("pip install pandas")

sandbox.kill()

In [ ]:
# --- Verified: subprocess + RLIMIT_CPU actually stops a runaway CPU loop ---
import resource
import subprocess
import time

def limit_cpu():
    # preexec_fn runs in the child, after fork() and before exec() -- the limit applies
    # only to the subprocess, never to the parent process running this notebook.
    resource.setrlimit(resource.RLIMIT_CPU, (2, 2))  # 2 CPU-seconds, (soft, hard)

# Case 1: well-behaved code runs and returns normally under the limit.
r1 = subprocess.run(["python3", "-c", "print(sum(range(1000)))"],
                     timeout=10, preexec_fn=limit_cpu, capture_output=True, text=True)
print("good_code:", r1.returncode, repr(r1.stdout.strip()))

# Case 2: a CPU-bound infinite loop -- the OS should SIGKILL it once it burns 2 CPU-seconds,
# well before the outer timeout=10 would otherwise fire.
start = time.time()
r2 = subprocess.run(["python3", "-c", "while True: pass"],
                     timeout=10, preexec_fn=limit_cpu, capture_output=True, text=True)
elapsed = time.time() - start
print(f"bad_code (infinite loop): returncode={r2.returncode} elapsed={elapsed:.2f}s")
# Verified real output: returncode=-24 (killed by SIGXCPU) after ~2.0s. This part
# genuinely works and is cross-platform.

In [ ]:
# --- Verified: RLIMIT_AS (memory cap) is NOT reliably enforceable cross-platform ---
# This is exactly why "subprocess + resource limits" must never be treated as a real
# security boundary -- even the parts of it that look like they should work uniformly
# can silently fail on a given platform.
import resource

try:
    resource.setrlimit(resource.RLIMIT_AS, (256 * 1024 * 1024,) * 2)  # attempt: cap at 256MB
    print("RLIMIT_AS set successfully")
except ValueError as e:
    print("RLIMIT_AS failed on this platform:", e)
# Verified real output on macOS: "ValueError: current limit exceeds maximum limit" --
# the memory cap silently does not apply here, while the CPU cap above does.